In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

In [ ]:
import equinox as eqx
import esm  # pip install fair-esm==2.0.0
import esm2quinox
import jax.random as jrandom
import optax  # pip install optax
from surr_model.dataset.dataset import Dataset_PEPBI
from surr_model.functions.model import (
    stripped_PREDICTOR,
)
from surr_model.functions.training import (
    eval_step,
    train_model_validation,
)
import jax.numpy as jnp
import jax.random as jr
import jax
from torch.utils.data import DataLoader
import pickle


In [4]:
base_path = "/home/kunzj"
# base_path = '/home/jkunz/master_uni_hd/internship_amsterdam'

mmpbsa dataset - cosine similarity

In [5]:
# data_mmpbsa_all = Dataset_PEPBI(
#     columns=["seq_prot", "seq_pept", "final_score"],
#     data_path=f"{base_path}/BindCraft_uva_internship/data/mmpbsa/finished_data_cosine_sim.csv",
#     transform=esm2quinox.tokenise,
# )
# data_mmpbsa_train_two_split = Dataset_PEPBI(
#     columns=["seq_prot", "seq_pept", "final_score"],
#     data_path=f"{base_path}/BindCraft_uva_internship/data/mmpbsa/two_way_split/finished_data_cosine_sim_train.csv",
#     transform=esm2quinox.tokenise,
# )
# data_mmpbsa_validation_two_split = Dataset_PEPBI(
#     columns=["seq_prot", "seq_pept", "final_score"],
#     data_path=f"{base_path}/BindCraft_uva_internship/data/mmpbsa/two_way_split/finished_data_cosine_sim_validation.csv",
#     transform=esm2quinox.tokenise,
# )

data_mmpbsa_train_three_split = Dataset_PEPBI(
    columns=["seq_prot", "seq_pept", "final_score"],
    data_path=f"{base_path}/BindCraft_uva_internship/data/mmpbsa/three_way_split/finished_data_cosine_sim_train.csv",
    transform=esm2quinox.tokenise,
)
data_mmpbsa_validation_three_split = Dataset_PEPBI(
    columns=["seq_prot", "seq_pept", "final_score"],
    data_path=f"{base_path}/BindCraft_uva_internship/data/mmpbsa/three_way_split/finished_data_cosine_sim_validation.csv",
    transform=esm2quinox.tokenise,
)
data_mmpbsa_test_three_split = Dataset_PEPBI(
    columns=["seq_prot", "seq_pept", "final_score"],
    data_path=f"{base_path}/BindCraft_uva_internship/data/mmpbsa/three_way_split/finished_data_cosine_sim_test.csv",
    transform=esm2quinox.tokenise,
)


# loader_mmpbsa_all = DataLoader(data_mmpbsa_all, batch_size=1)
# loader_mmpbsa_train_two_split = DataLoader(data_mmpbsa_train_two_split, batch_size=3)
# loader_mmpbsa_validation_two_split = DataLoader(
#     data_mmpbsa_validation_two_split, batch_size=1
# )


loader_mmpbsa_train_three_split = DataLoader(
    data_mmpbsa_train_three_split, batch_size=3
)
loader_mmpbsa_validation_three_split = DataLoader(
    data_mmpbsa_validation_three_split, batch_size=1
)
loader_mmpbsa_test_three_split = DataLoader(data_mmpbsa_test_three_split, batch_size=1)

ppi affinity dataset - cosine similarity


In [6]:
# # two way
# data_ppi_train_two_split = Dataset_PEPBI(
#     columns=["Prot_Seq", "Pept_Seq", "Energy"],
#     data_path=f"{base_path}/BindCraft_uva_internship/data/ppi_affinity_dataset/cosine_sim_scaled/two_way/train.csv",
#     transform=esm2quinox.tokenise,
# )
# data_ppi_validation_two_split = Dataset_PEPBI(
#     columns=["Prot_Seq", "Pept_Seq", "Energy"],
#     data_path=f"{base_path}/BindCraft_uva_internship/data/ppi_affinity_dataset/cosine_sim_scaled/two_way/validation.csv",
#     transform=esm2quinox.tokenise,
# )
# #  modified two way
# data_ppi_train_two_split_modified = Dataset_PEPBI(
#     columns=["Prot_Seq", "Pept_Seq", "Energy"],
#     data_path=f"{base_path}/BindCraft_uva_internship/data/ppi_affinity_dataset/cosine_sim_scaled/two_way_modified/train.csv",
#     transform=esm2quinox.tokenise,
# )
# data_ppi_validation_two_split_modified = Dataset_PEPBI(
#     columns=["Prot_Seq", "Pept_Seq", "Energy"],
#     data_path=f"{base_path}/BindCraft_uva_internship/data/ppi_affinity_dataset/cosine_sim_scaled/two_way_modified/validation.csv",
#     transform=esm2quinox.tokenise,
# )
# three way
data_ppi_train_three_split = Dataset_PEPBI(
    columns=["Prot_Seq", "Pept_Seq", "Energy"],
    data_path=f"{base_path}/BindCraft_uva_internship/data/ppi_affinity_dataset/cosine_sim_scaled/three_way/train.csv",
    transform=esm2quinox.tokenise,
)
data_ppi_validation_three_split = Dataset_PEPBI(
    columns=["Prot_Seq", "Pept_Seq", "Energy"],
    data_path=f"{base_path}/BindCraft_uva_internship/data/ppi_affinity_dataset/cosine_sim_scaled/three_way/validation.csv",
    transform=esm2quinox.tokenise,
)
data_ppi_test_three_split = Dataset_PEPBI(
    columns=["Prot_Seq", "Pept_Seq", "Energy"],
    data_path=f"{base_path}/BindCraft_uva_internship/data/ppi_affinity_dataset/cosine_sim_scaled/three_way/test.csv",
    transform=esm2quinox.tokenise,
)

# # loaders
# loader_ppi_train_two_split = DataLoader(data_ppi_train_two_split, batch_size=20)
# loader_ppi_validation_two_split = DataLoader(
#     data_ppi_validation_two_split, batch_size=1
# )

# loader_ppi_train_two_split_modified = DataLoader(
#     data_ppi_train_two_split_modified, batch_size=20
# )
# loader_ppi_validation_two_split_modified = DataLoader(
#     data_ppi_validation_two_split_modified, batch_size=1
# )

loader_ppi_train_three_split = DataLoader(data_ppi_train_three_split, batch_size=10)
loader_ppi_validation_three_split = DataLoader(
    data_ppi_validation_three_split, batch_size=1
)
loader_ppi_test_three_split = DataLoader(data_ppi_test_three_split, batch_size=1)


### train on three way ppi dataset

In [7]:
eqx.clear_caches()
jax.clear_caches()
# generating keys
model_key, call_key = jr.split(jrandom.PRNGKey(0), 2)

# initializing models
torch_model, _ = esm.pretrained.esm2_t30_150M_UR50D()
model_esm2 = esm2quinox.from_torch(torch_model)

model_aff = stripped_PREDICTOR(
    model_prot=model_esm2,model_pept=model_esm2, key=model_key
)

# initializing optimizer
optim = optax.adam(learning_rate=0.001)
opt_state = optim.init(eqx.filter(model_aff, eqx.is_inexact_array))
best_model_first, train_losses_first, val_losses_first = (
    train_model_validation(
        training_DataLoader=loader_ppi_train_three_split,
        validation_Dataloader=loader_ppi_validation_three_split,
        max_epochs=30,
        model_aff=model_aff,
        opt_state=opt_state,
        optim=optim,
        key=call_key,
    )
)

test_key_init = jrandom.PRNGKey(1)
inference_model = eqx.nn.inference_mode(best_model_first)

test_loss_list = []
pred_y_list = []
y_val_list = []

for entry, (x_prot, x_pept, y_val) in enumerate(loader_ppi_test_three_split):
    x_prot, x_pept, y_val = jnp.array(x_prot), jnp.array(x_pept), jnp.array(y_val)
    test_key = jr.split(test_key_init, x_pept.shape[0])
    test_loss, pred_y = eval_step(inference_model, x_prot, x_pept, y_val, test_key)
    test_loss_list.append(test_loss.item())
    pred_y_list.append(pred_y.item())
    y_val_list.append(y_val.item())
    print(
        f"[Entry {entry + 1}] , Test Loss: {test_loss:.3f}, Predicted Y: {pred_y.item():.3f}, , True Y: {y_val.item():.3f}"
    )

results_ppi_three_way = {
    "test_loss_list": test_loss_list,
    "pred_y_list": pred_y_list,
    "y_val_list": y_val_list,
    "train_losses": train_losses_first,
    "val_losses": val_losses_first,
}

with open("/home/kunzj/BindCraft_uva_internship/surr_model/training_results_jax/results_ppi_three_way.pkl", "wb") as f:
    pickle.dump(results_ppi_three_way, f)


Epochs:   0%|          | 0/30 [00:00<?, ?it/s]2025-12-16 19:54:41.331387: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-12-16 19:55:11.668993: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-12-16 19:55:22.811582: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
Epochs:   3%|▎ 

[Epoch 1] Train Loss: 0.162384, Val Loss: 0.004851
	(New best model saved at Epoch: 1. )


Epochs:   7%|▋         | 2/30 [01:22<18:28, 39.58s/it]

[Epoch 2] Train Loss: 0.125245, Val Loss: 0.011490


2025-12-16 19:56:18.782967: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.54GiB (rounded to 1653371648)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-12-16 19:56:18.783524: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] *************************************************************************************_______________
E1216 19:56:18.783562  463246 pjrt_stream_executor_client.cc:2916] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1653371480 bytes. [tf-allocator-allocation-error='']


ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1653371480 bytes.

### train model on three way mmpbsa

In [ ]:
eqx.clear_caches()
jax.clear_caches()
# generating keys
model_key, call_key = jr.split(jrandom.PRNGKey(0), 2)

# initializing models
torch_model, _ = esm.pretrained.esm2_t30_150M_UR50D()
model_esm2 = esm2quinox.from_torch(torch_model)

model_aff = stripped_PREDICTOR(
    model_prot=model_esm2,model_pept=model_esm2, key=model_key
)

# initializing optimizer
optim = optax.adam(learning_rate=0.001)
opt_state = optim.init(eqx.filter(model_aff, eqx.is_inexact_array))
best_model_first, train_losses_first, val_losses_first = (
    train_model_validation(
        training_DataLoader=loader_mmpbsa_train_three_split,
        validation_Dataloader=loader_mmpbsa_validation_three_split,
        max_epochs=30,
        model_aff=model_aff,
        opt_state=opt_state,
        optim=optim,
        key=call_key,
    )
)

test_key_init = jrandom.PRNGKey(1)
inference_model = eqx.nn.inference_mode(best_model_first)


test_loss_list = []
pred_y_list = []
y_val_list = []

for entry, (x_prot, x_pept, y_val) in enumerate(loader_mmpbsa_test_three_split):
    x_prot, x_pept, y_val = jnp.array(x_prot), jnp.array(x_pept), jnp.array(y_val)
    test_key = jr.split(test_key_init, x_pept.shape[0])
    test_loss, pred_y = eval_step(inference_model, x_prot, x_pept, y_val, test_key)
    test_loss_list.append(test_loss.item())
    pred_y_list.append(pred_y.item())
    y_val_list.append(y_val.item())
    print(
        f"[Entry {entry + 1}] , Test Loss: {test_loss:.3f}, Predicted Y: {pred_y.item():.3f}, , True Y: {y_val.item():.3f}"
    )

results_mmpbsa_three_way = {
    "test_loss_list": test_loss_list,
    "pred_y_list": pred_y_list,
    "y_val_list": y_val_list,
    "train_losses": train_losses_first,
    "val_losses": val_losses_first,
}
with open("/home/kunzj/BindCraft_uva_internship/surr_model/training_results_jax/results_mmpbsa_three_way.pkl", "wb") as f:
    pickle.dump(results_mmpbsa_three_way, f)

Epochs:   0%|          | 0/30 [00:00<?, ?it/s]2025-12-16 13:12:25.737763: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-12-16 13:12:39.694795: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-12-16 13:12:39.695214: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-12-16 13:1

[Epoch 1] Train Loss: 0.203725, Val Loss: 0.922988
	(New best model saved at Epoch: 1. )


Epochs:   7%|▋         | 2/30 [00:29<05:42, 12.24s/it]

[Epoch 2] Train Loss: 0.142738, Val Loss: 0.510193


Epochs:  10%|█         | 3/30 [00:29<03:05,  6.86s/it]

...

[Epoch 28] Train Loss: 0.008660, Val Loss: 0.744051


Epochs:  97%|█████████▋| 29/30 [00:41<00:00,  2.20it/s]

[Epoch 29] Train Loss: 0.007804, Val Loss: 0.744528


[Epoch 30] Train Loss: 0.006982, Val Loss: 0.723376
Training complete.


[Entry 1] , Test Loss: 0.460, Predicted Y: -0.150, , True Y: -0.828
[Entry 2] , Test Loss: 0.347, Predicted Y: -0.055, , True Y: -0.644
[Entry 3] , Test Loss: 0.232, Predicted Y: 0.130, , True Y: -0.352
[Entry 4] , Test Loss: 0.006, Predicted Y: 0.258, , True Y: 0.183
[Entry 5] , Test Loss: 0.998, Predicted Y: 0.001, , True Y: 1.000
[Entry 6] , Test Loss: 0.009, Predicted Y: 0.278, , True Y: 0.185
[Entry 7] , Test Loss: 0.060, Predicted Y: 0.275, , True Y: 0.031
[Entry 8] , Test Loss: 0.031, Predicted Y: 0.548, , True Y: 0.371
[Entry 9] , Test Loss: 0.103, Predicted Y: 0.136, , True Y: -0.184
[Entry 10] , Test Loss: 0.140, Predicted Y: -0.165, , True Y: -0.540
[Entry 11] , Test Loss: 0.244, Predicted Y: 0.172, , True Y: -0.323
[Entry 12] , Test Loss: 0.196, Predicted Y: 0.287, , True Y: 0.730
[Entry 13] , Test Loss: 0.091, Predicted Y: 0.052, , True Y: -0.250
[Entry 14] , Test Loss: 0.159, Predicted Y: 0.026, , True Y: -0.372
[Entry 15] , Test Loss: 0.027, Predicted Y: 0.687, , True Y:

### train model on three way ppi and then three way mmpbsa

In [ ]:
eqx.clear_caches()
jax.clear_caches()
# generating keys
model_key, call_key = jr.split(jrandom.PRNGKey(0), 2)

# initializing models
torch_model, _ = esm.pretrained.esm2_t30_150M_UR50D()
model_esm2 = esm2quinox.from_torch(torch_model)

model_aff = stripped_PREDICTOR(
    model_prot=model_esm2,model_pept=model_esm2, key=model_key
)

# initializing optimizer
optim = optax.adam(learning_rate=0.001)
opt_state = optim.init(eqx.filter(model_aff, eqx.is_inexact_array))
best_model_first, train_losses_first, val_losses_first = (
    train_model_validation(
        training_DataLoader=loader_ppi_train_three_split,
        validation_Dataloader=loader_ppi_validation_three_split,
        max_epochs=30,
        model_aff=model_aff,
        opt_state=opt_state,
        optim=optim,
        key=call_key,
    )
)

# initializing optimizer
optim = optax.adam(learning_rate=0.001)
opt_state = optim.init(eqx.filter(best_model_first, eqx.is_inexact_array))
best_model_second, train_losses_second, val_losses_second= (
    train_model_validation(
        training_DataLoader=loader_mmpbsa_train_three_split,
        validation_Dataloader=loader_mmpbsa_validation_three_split,
        max_epochs=10,
        model_aff=best_model_first,
        opt_state=opt_state,
        optim=optim,
        key=call_key,
    )
)


test_key_init = jrandom.PRNGKey(1)
inference_model = eqx.nn.inference_mode(best_model_second)

test_loss_list = []
pred_y_list = []
y_val_list = []

for entry, (x_prot, x_pept, y_val) in enumerate(loader_mmpbsa_test_three_split):
    x_prot, x_pept, y_val = jnp.array(x_prot), jnp.array(x_pept), jnp.array(y_val)
    test_key = jr.split(test_key_init, x_pept.shape[0])
    test_loss, pred_y = eval_step(inference_model, x_prot, x_pept, y_val, test_key)
    test_loss_list.append(test_loss.item())
    pred_y_list.append(pred_y.item())
    y_val_list.append(y_val.item())
    print(
        f"[Entry {entry + 1}] , Test Loss: {test_loss:.3f}, Predicted Y: {pred_y.item():.3f}, , True Y: {y_val.item():.3f}"
    )

results_ppi_two_way_mmpbsa_three_way = {
    "test_loss_list": test_loss_list,
    "pred_y_list": pred_y_list,
    "y_val_list": y_val_list,
    "train_losses_first": train_losses_first,
    "val_losses_first": val_losses_first,
    "train_losses_second": train_losses_second,
    "val_losses_second": val_losses_second,
}

with open("/home/kunzj/BindCraft_uva_internship/surr_model/training_results_jax/results_ppi_two_way_mmpbsa_three_way.pkl", "wb") as f:
    pickle.dump(results_ppi_two_way_mmpbsa_three_way, f)

Epochs:   0%|          | 0/30 [00:00<?, ?it/s]2025-12-16 15:53:45.347012: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-12-16 15:54:15.767917: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-12-16 15:54:27.779383: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
Epochs:   3%|▎ 

[Epoch 1] Train Loss: 0.126960, Val Loss: 0.001339
	(New best model saved at Epoch: 1. )


Epochs:   7%|▋         | 2/30 [01:06<13:56, 29.89s/it]

[Epoch 2] Train Loss: 0.094204, Val Loss: 0.006474
	(New best model saved at Epoch: 2. )


Epochs:  10%|█         | 3/30 [01:20<10:17, 22.86s/it]

...

Epochs:  90%|█████████ | 9/10 [00:31<00:00,  1.05it/s]

[Epoch 9] Train Loss: 0.030312, Val Loss: 0.779970
	(New best model saved at Epoch: 9. )


[Epoch 10] Train Loss: 0.022799, Val Loss: 0.800839
	(New best model saved at Epoch: 10. )
Training complete.
[Entry 1] , Test Loss: 0.328, Predicted Y: -0.255, , True Y: -0.828
[Entry 2] , Test Loss: 0.216, Predicted Y: -0.179, , True Y: -0.644
[Entry 3] , Test Loss: 0.365, Predicted Y: 0.252, , True Y: -0.352
[Entry 4] , Test Loss: 0.017, Predicted Y: 0.313, , True Y: 0.183
[Entry 5] , Test Loss: 0.990, Predicted Y: 0.005, , True Y: 1.000
[Entry 6] , Test Loss: 0.003, Predicted Y: 0.128, , True Y: 0.185
[Entry 7] , Test Loss: 0.049, Predicted Y: 0.252, , True Y: 0.031
[Entry 8] , Test Loss: 0.111, Predicted Y: 0.705, , True Y: 0.371
[Entry 9] , Test Loss: 0.081, Predicted Y: 0.101, , True Y: -0.184
[Entry 10] , Test Loss: 0.092, Predicted Y: -0.236, , True Y: -0.540
[Entry 11] , Test Loss: 0.209, Predicted Y: 0.135, , True Y: -0.323
[Entry 12] , Test Loss: 0.124, Predicted Y: 0.378, , True Y: 0.730
[Entry 13] , Test Loss: 0.059, Predicted Y: -0.007, , True Y: -0.250
[Entry 14] , Test

[Entry 20] , Test Loss: 0.003, Predicted Y: -0.197, , True Y: -0.251
[Entry 21] , Test Loss: 0.025, Predicted Y: 0.569, , True Y: 0.409
[Entry 22] , Test Loss: 0.204, Predicted Y: 0.529, , True Y: 0.077
[Entry 23] , Test Loss: 0.184, Predicted Y: 0.126, , True Y: -0.303
[Entry 24] , Test Loss: 0.399, Predicted Y: 0.124, , True Y: -0.507
[Entry 25] , Test Loss: 0.366, Predicted Y: 0.136, , True Y: -0.469
[Entry 26] , Test Loss: 0.495, Predicted Y: 0.130, , True Y: 0.834
[Entry 27] , Test Loss: 0.104, Predicted Y: -0.044, , True Y: -0.367
[Entry 28] , Test Loss: 0.000, Predicted Y: -0.172, , True Y: -0.166
[Entry 29] , Test Loss: 0.008, Predicted Y: 0.631, , True Y: 0.723
[Entry 30] , Test Loss: 0.068, Predicted Y: -0.216, , True Y: -0.477
[Entry 31] , Test Loss: 0.328, Predicted Y: -0.275, , True Y: -0.848
[Entry 32] , Test Loss: 0.041, Predicted Y: 0.276, , True Y: 0.073
[Entry 33] , Test Loss: 0.772, Predicted Y: 0.204, , True Y: -0.675


: 